<a href="https://colab.research.google.com/github/AnNguyenGia/CIFAR-10/blob/main/Cifar_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import PIL
import torchvision.models as models
import torch.nn as nn
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim

In [28]:
transform = transforms.Compose([
    transforms.Resize((224, 224)), # Scales height and width to 224x224
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

#import datasets

In [ ]:
df_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
df_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:02<00:00, 65.4MB/s]


In [ ]:
classes = df_train.classes

# CNN strategy

In [ ]:
EPOCHS = 15
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 5, padding=2) # 32x32 -> 32x32 -> Pool -> 16x16
        self.conv2 = nn.Conv2d(16, 32, 5, padding=2) # 16x16 -> 16x16 -> Pool -> 8x8
        self.conv3 = nn.Conv2d(32, 64, 5, padding=2) # 8x8 -> 8x8 -> Pool -> 4x4

        self.pool = nn.MaxPool2d(2, 2)
        self.fla = nn.Flatten()


        self.fc1 = nn.Linear(in_features=50176, out_features=1000)
        self.fc2 = nn.Linear(in_features=1000, out_features=120)
        self.fc3 = nn.Linear(in_features=120, out_features=84)
        self.fc4 = nn.Linear(in_features=84, out_features=10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = self.fla(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)

        return x



In [ ]:
from tqdm.auto import tqdm
import torch
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Train on device: {device}")

model = CNN().to(device)

optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
trainloader = DataLoader(df_train, batch_size=32, shuffle=True)

patience = 3
epochs_no_improve = 0
best_loss = float('inf')

for epoch in range(EPOCHS):
    pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}")

    epoch_loss = 0

    for image, label in pbar:

        image = image.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        output = model(image)
        loss = F.cross_entropy(output, label)
        loss.backward()
        optimizer.step()

        pbar.set_postfix(loss=loss.item())
        epoch_loss += loss.item()


    avg_loss = epoch_loss / len(trainloader)

    if avg_loss < best_loss:
        best_loss = avg_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"\nWarning: {epochs_no_improve}/{patience}")

        if epochs_no_improve >= patience:
            print(f"Early Stopping at Epoch {epoch+1}.")
            break

Đang train trên thiết bị: cpu


Epoch 1:   0%|          | 0/1563 [00:00<?, ?it/s]

In [ ]:

model.eval()

test_loss = 0.0
correct = 0
total = 0


with torch.no_grad():
    for image, label in df_test:

        image = image.unsqueeze(0).to(device)

        if not isinstance(label, torch.Tensor):
            target = torch.tensor([label]).to(device)
        else:
            target = label.unsqueeze(0).to(device)


        output = model(image)


        loss = nn.CrossEntropyLoss(output, target)
        test_loss += loss.item()


        predicted = torch.argmax(output, dim=1)
        if predicted.item() == label:
            correct += 1

        total += 1

--
avg_test_loss = test_loss / total
accuracy = 100 * correct / total

print("-" * 30)

print(f"Loss: {avg_test_loss:.4f}")
print(f"Accuracy: {accuracy:.2f}% ({correct}/{total} images)")
print("-" * 30)

tensor([[ -2.4805, -16.3471,   0.8312,  24.2464,  -5.5784,  19.9826, -12.6956,
          -0.5310,   0.9926, -17.1356]], device='cuda:0',
       grad_fn=<AddmmBackward0>)

#Transfer Learning strategy (VGGNet)

In [29]:
import torch.nn as nn
from torchvision import models

# 1. Load the pre-trained VGG16 model
vggnet = models.vgg16(pretrained=True)

# 2. Freeze the Feature Extraction part (Convolutional layers)
# This prevents the pre-trained weights from being updated during training
for param in vggnet.features.parameters():
    param.requires_grad = False

# 3. Get the input dimension for the first linear layer in the classifier
# For VGG16, this value is typically 25088 (512 * 7 * 7)
input_from_features = vggnet.classifier[0].in_features

# 4. Redefine the Classifier with your custom architecture
# Architecture flow: 25088 -> 4096 -> 120 -> 240 -> 120 -> 60 -> 10
vggnet.classifier = nn.Sequential(
    # First Layer: Transition from Features to 4096
    nn.Linear(input_from_features, 4096),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.5),

    # Custom hidden layers as requested
    nn.Linear(4096, 120),
    nn.ReLU(inplace=True),

    nn.Linear(120, 240),
    nn.ReLU(inplace=True),

    nn.Linear(240, 120),
    nn.ReLU(inplace=True),

    nn.Linear(120, 60),
    nn.ReLU(inplace=True),

    # Final Output Layer: 10 target classes
    nn.Linear(60, 10)
)

# 5. Push the model to the target device (GPU or CPU)
vggnet = vggnet.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:


# 1. Configuration
EPOCHS_vgg = 20
patience = 5
best_loss = float('inf')
epochs_no_improve = 0

# 2. Loss Function and Optimizer
# Note: Only pass classifier parameters to the optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vggnet.classifier.parameters(), lr=0.0001)

# 3. Training Loop
for epoch in range(EPOCHS_vgg):
    vggnet.train()  # Set model to training mode
    epoch_loss = 0.0

    # Progress bar for the current epoch
    pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{EPOCHS_vgg}")

    for images, labels in pbar:
        # Move data to the same device as the model
        images = images.to(device)
        labels = labels.to(device)

        # Reset gradients from the previous step
        optimizer.zero_grad()

        # Forward pass: Compute predicted outputs by passing inputs to the model
        outputs = vggnet(images)

        # Calculate the batch loss
        loss = criterion(outputs, labels)

        # Backward pass: Compute gradient of the loss with respect to model parameters
        loss.backward()

        # Perform a single optimization step (parameter update)
        optimizer.step()

        # Statistics
        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    # --- End of Epoch Logic ---

    # Calculate average loss for the entire epoch
    avg_loss = epoch_loss / len(trainloader)
    print(f"\nEpoch {epoch+1} Summary - Average Loss: {avg_loss:.4f}")

    # 4. Early Stopping Check
    if avg_loss < best_loss:
        # Improvement found
        best_loss = avg_loss
        epochs_no_improve = 0
        # Optional: Save the best model weights
        # torch.save(vggnet.state_dict(), 'best_vgg_model.pth')
        print(f"Model improved. Saving checkpoint...")
    else:
        # No improvement
        epochs_no_improve += 1
        print(f"No improvement in loss for {epochs_no_improve} epoch(s).")

        if epochs_no_improve >= patience:
            print(f"Early Stopping triggered. Training halted.")
            break

Epoch 1/20:   0%|          | 0/1563 [00:00<?, ?it/s]